# Module 09 — Lecture: Seismic catalogues. Catalogue preprocessing (I)

**DIGIHAZ PhD Course** | Disaster Risk Reduction

---


## Learning Objectives

By the end of this session you will be able to:

1. Acquire and load seismic catalogues.
2. Identify and remove duplicates in the catalogue.
3. Homogenise a seismic catalogue.

## Introduction
The **seismic catalogues** contain information about the earthquakes that have been registered during a certain period of time in a given area. Depending on the agency, country or intended use of the catalogue the information contained in these can vary. Nevertheless, a catalogue is expected to provide information on the **location** (longitude, latitude, depth), **size** of the earthquake (magnitude and/or intensity), **time** of ocurrence (date and time), IDs of the events, etc.

It is not uncommon to find that the earthquakes are expressed using different magnitude scales (**mB**, **mblg**, **Mw**, **mL**,...), especially when the catalogue covers wide areas (more probability of covering different seismic stations) and extended periods of time (reflecting the evolution of the local seismic network). Or even, with missing information (older records, network malfunctions) or duplicities (when country-wise bulletins use information from regional seismic networks). Hence, the need for a preprocessing stage of the catalogues. This preprocessing covers mainly two aspects: elimination of erroneous entries and duplicates, and homogenisation of the catalogue (express all the earthquakes using a common magnitude scale, in this case moment magnitude Mw).

For this exercise we will use the catalogue of South-eastern Spain (IGN, 2022; https://www.ign.es/web/ign/portal/sis-catalogo-terremotos), covering the period from 1371 up until May 2026. This way, we will have both the historical (pre-instrumental era and the instrumental era of the seismic catalogue. You can download this catalogue by 

## Preprocessing

There are three main procedures that are normally included inside the preprocessing stage:

1) Removal of duplicates and erroneous entries.
2) Homogenisation
3) Declustering 

In this exercise we will approach the first two stages, and the third will be dealt with in the next exercise.

### Importing and installing the Python libraries

The first step towards the preprocessing with python is installing and loading all the modules that will be used:

In [ ]:
pip install pandas

In [ ]:
import pandas as pd                    # Standard data analysis library.
import numpy as np                     # Vectorized calculus library.

### Removal of duplicates

Once these are loaded the functions to be used in the preprocessing should be defined. We will start by defining an ID assigning function. This function will create unique identifers for each event based in a set of fields contained in the catalogue. Before we do that it is important to open the catalogue with any txt or spreadsheet editor to see which information is available.

Event|Date|Time|Latitude|Longitude|Depth|Intensity|Magnitude|Magnitude_type|Location

Event: string of text
Date: dd/mm/yyyy
Time: hh:mm:ss
Latitude: xx.xxxx
Longitude: x.xxxx
Depth: in km
Intensity: roman numerals (I to X)
Magnitude: float
Magnitude_type: there is 7 different types (1,2,3,4,5,6 and 13)
Location: Municipalities

All the columns are separated by several whitespaces (and even some columns have several whitespaces before and after the data, i.e. the Intensity column). 
In the case of Spain it is possible to create a unique ID by concatenating the Date and Time fields.

In [ ]:
data = pd.read_csv('catalogue.csv', sep=';', skipinitialspace=True)
# With the instruction skipinitialspace=True we avoid the whitespaces
# inside each column.
display(data) # Then we can display de data as a table.

In [ ]:
def id_creator(data:pd.DataFrame, ID_name:str,
               field_name_list:list[str]) -> pd.DataFrame:
    """
    Creates IDs for each event in the catalogue.

    Parameters
    ----------
    data : PANDAS DATAFRAME
        Seismic catalogue loaded using pandas.
    ID_name: STR
        Name for field which will contain the created ID
    field_name_list : LIST[STR]
        List of names of the the fields to be used to
        generate the unique ID.

    Returns
    -------
    data : PANDAS DATAFRAME
        Catalogue including a unique ID field with name "ID_name"

    """
        
    data[ID_name] = data[field_name_list].astype(str).agg(''.join, axis=1)
    

This instruction reads only the columns with names contained in the list field_name_list. Then converts the data to strings (text) with the function .astype(str) and finally concatenates these texts using .agg(''.join, axis=1). We could insert '\_' after each field data (i.e. 11/01/2001_11:22:00) by inserting the low dash before the join function ('_'.join, axis=1) or the character we wanted to, but it is best not to overcomplicate the solution or increase memory requirements if possible. 

We can apply it now by using the fields 'Date', 'Time', 'Latitude' and 'Longitude'

In [ ]:
id_creator(data, 'ID', ['Date', 'Time', 'Longitude', 'Latitude'])

In [ ]:
display(data)


Each event has now a unique ID that can be used to identify duplicates in the catalogue. To do this pandas already has a built-in function, .drop_duplicates().

In [ ]:
# Before droping the duplicates we will gather how many 
#events are in our catalogue.
lenght_o = int(len(data))
# We can see the duplicates to check whether the algorithm
# is working as inteded.
display(data[data.duplicated(subset=['ID'], keep=False)])

In [ ]:
# Then the duplicates are filtered out:
data = data.drop_duplicates(subset=['ID'])
# The subset argument indicates which column is used to look 
#for duplicates. It is important that we assing a variable to
#this action, otherwise the drop_duplicates operation will
#not be recorded.
lenght_a = int(len(data))
print(lenght_o-lenght_a, 'duplicates were found.')

**Proposed exercise 1) How would we recover a list with the duplicate records in the catalogue? HINT: consider creating a duplicate with the original catalogue in order to compare with the filtered catalogue. There are some functions and operators in base python and pandas that can be used to compare two sets of data.**

### Homogenisation process 
The next step is to convert the roman numeral intensity (I, II, IV, etc.)  into a float value (1.0, 2.0, ...). There are several ways to achieve this, one of the most common ones is to use dictionaries.

The dictionaries relate one value to another through key-value pairs (using : to separate one from another). Example:

dictionary = {'I':1.0, 'I-II':1.5}

So when we call it and gets the value 'I' it will output the value 1.0.

value = dictionary('I')

We will define then a function that converts such numerals into floats:

In [ ]:
def int_converter(data:pd.DataFrame) -> pd.DataFrame:
    """
    Converts intensity in roman numerals (STR) into FLOAT.
    
    Parameters
    ----------
    data : PANDAS DATAFRAME
           Seismic catalogue. Minimum columns needed for 
           this function: 'Intensity'.
           Intensity: STR field containing the roman numerals of 
           the intensity without whitespaces in the cells. 

    Returns
    -------
    data : PANDAS DATAFRAME
           The seismic catalogues with an additional "I_num"
           column containing the float value of the intensity of
           the earthquake.
    """
    
    mapping = {
        'I':1.0, 'I-II':1.5, 'II':2.0, 'II-III':2.5, 'III':3.0,
        'III-IV':3.5, 'IV':4.0, 'IV-V':4.5, 'V':5.0, 'V-VI':5.5,
        'VI':6.0, 'VI-VII':6.5, 'VII':7.0, 'VII-VIII':7.5,
        'VIII':8.0, 'VIII-IX':8.5, 'IX':9.0, 'IX-X':9.5,
        'X':10.0, 'X-XI':10.5, 'XI':11.0, 'XI-XII':11.5, 'XII':12.0
    }

    data['I_num'] = data['Intensity'].map(mapping).fillna(-1)
    # with the map function .map() we pass as an argument 
    #the dictionary previously defined. Then, with .fillna()
    # we can fill the 'None' values with the desired value.
    # This last part is optional, we could just omit it if we wanted.

In [ ]:
int_converter(data)

In [ ]:
display(data)

**Proposed exercise 2) Could you think of another method/scheme to convert the roman numerals into float values. HINT: one method could rely on the use of 'regular expressions'.**

Moving on to the homogenisation process it is worth noting it is not trivial. To perform this preprocessing task we need conversion equations. These equations must be defined by using earthquakes expressed in moment magnitude and other local magnitude scales.

In the case of Spain, in the latest Seismic Hazard Map update (IGN-UPM Working Group, 2013), four equations were defined to convert different magnitude and intensity scales into moment magnitude (Mw):

|y = a + b x | Range | Magnitude type |
| :--- | :--- | :---:|
| $M_w$ = 1.6560 + 0.545 $I_{max}$ | 3.0 - 9.5 | 1 |
| $M_w$ = 0.2900 + 0.973 $m_{bLg}$ | 3.1 - 7.3 | 2 |
| $M_w$ = -1.528 + 1.213 $m_b$     | 3.7 - 6.3 | 3 |
| $M_w$ = 0.6760 + 0.836 $m_{bLg}$ | 3.0 - 5.1 | 4 |

With these equations it is possible to define a function to convert each local
scale into moment magnitude.

In [ ]:
def mw_converter(data:pd.DataFrame) -> pd.DataFrame:
    """
    Converts the intensity and local magnitude scales into 
    moment magnitude (Mw) using the equations shown in
    the publication "Actualización de mapas de peligrosidad sísmica
    de España" 2012 (p.27).
    
    Parameters
    ----------
    data : PANDAS DATAFRAME
           Seismic catalogue. Minimum columns required for this script:
           I_num: Numerical intensity converted from roman numeral.
           Magnitud: Magnitude (local or moment magnitude) as a float.
           Magnitude_Type: Magnitude type (numerical code from
           1 to 6 (and 13))

    Returns
    -------
    data : PANDAS DATAFRAME
           Modified catalogue with a column "mw" containing the
           converted moment magnitude of the earthquakes.

    """

    # We transform then the table into a structured dictionary 
    #having the following columns:
    # eq = magnitude_type: {ordinate, slope, min_value,
    #max_value, column used}
    eqs = {
        1: {"a": 1.6560, "b": 0.545, "min": 3.0, "max": 9.5,
            "col": 'I_num'},
        2: {"a": 0.2900, "b": 0.973, "min": 3.1, "max": 7.3,
            "col": "Magnitude"},
        3: {"a": -1.528, "b": 1.213, "min": 3.7, "max": 6.3,
            "col": "Magnitude"},
        4: {"a": 0.6760, "b": 0.836, "min": 3.0, "max": 5.1,
            "col": "Magnitude"},
    }
    
    # Initialize result, we chose -9999 as an absurd value
    #for the invalid entries that should be discarded. If the
    #algorithm does not overwrite the initial -9999 it means
    #it is erroneous.
    magnitudes = np.full(len(data), -9999.0)
    
    for type_m, eq in eqs.items():
        col = eq["col"]
        # In the following block we filter the data by type
        #and also we check that it is within the range of application.
        mask = (
            (data["Magnitude_type"] == type_m) &
            (data[col] >= eq["min"]) &
            (data[col] <= eq["max"])
        )
        
        magnitudes[mask] = eq["a"] + eq["b"] * data.loc[mask, col]
    
    # Handle Magnitude_types >= 5 where no conversion
    #should be applied.
    mask = data["Magnitude_type"] >= 5
    magnitudes[mask] = data.loc[mask, "Magnitude"]
    
    magnitudes = magnitudes.tolist()

    data["mw"] = magnitudes

Before applying the conversion we must locate those events which have no magnitude value and intensity value (mainly historical earthquake).

In [ ]:
# For the Mw conversion we should identify which events have no
# magnitude type nor intensity or magnitude values.

# We check which events have no magnitude type assigned.
c1 = data["Magnitude_type"].isna()
# Then we check which events have no intensity values.
c2 = data["Intensity"].isna()
# If they have no type assigned (c1) and no intensity (c2),
# then they will have type 0 which is not considered in
# the analysis (discarded event).
data.loc[c1 & c2, "Magnitude_type"] = 0
# Else, if they have no magnitude type assigned (c1) but they
# have an intensity value (~c2), then the type 1 will be assigned.
# The ~ operator negates the condition c2.
data.loc[c1 & ~c2, "Magnitude_type"] = 1

# Note that we are not checking if the magnitude field has a value,
# it is not necessary. If an event has magnitude and not magnitude type
# then we cannot assume the type for conversion, 
#so it should be discarded.

Now we can apply the moment magnitude conversion procedure. All the records with moment magnitude equal to -9999 can be discarded after.

In [ ]:
mw_converter(data)

In [ ]:
display(data)

In [ ]:
data = data[data.mw != -9999] # Filer out the erroneous entries.

In [ ]:
display(data)

### Decimal year computation

As a final task of this preprocessing stage we will convert the date and time into a numerical value (**decimal year**). This is widely used in several time-based algorithms, and also can be useful towards the representation of the seismic catalogue. It can be done with several different Python modules. In this exercise we will use the *Pandas* library as it is simple and effective.

If we observed the table containing our data we can see the format of the Date and Time fields, this should be beared in mind during the algorithm design stage.

In [ ]:
# We will combine the Date and Time fields in order 
#to construct a timestamp-like format. Then using the
#to_datetime() function inside Pandas we can input the 
#format of the timestamp.
dt = pd.to_datetime(
    data["Date"] + " " + data["Time"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

# Compute decimal year using pandas. First we obtain
#the year of each event.
year = dt.dt.year

start = pd.to_datetime(year.astype(str) + "-01-01")
end = pd.to_datetime((year + 1).astype(str) + "-01-01")
# We obtain the fraction of the year that has passed
#since the first day of the year for each event.
data["DecimalYear"] = year + (dt - start) / (end - start)

In [ ]:
display(data)

In [ ]:
data.to_csv('catalogue_2.csv')

## Conclusions and exercises

There is many other algorithms that can be used on the catalogue in order to prepare it for reasearch, these will depend on the characteristics of the seismicity in each region and/or period.

The following exercises are left for the reader:

1) How many earthquakes are deeper than 50 km in percentage? In the case the number is neglectable (around 5% at most) filter them out.
2) Normally we will not need accuracy on the moment magnitude greater than one or two decimal digits. Round the 'mw' field to one decimal.
3) Plot the catalogue (Moment magnitude vs Decimal year), you are free to change the colors/size of the markers depending on the magnitude. Do you notice something in the distribution of earthquakes after a big earthquake has ocurred (especially in the instrumental era of the catalogue, i.e. 1920 up until now)?

In the next exercise we will approach the computation of the seismic parameters and the declustering procedure on the catalogue. Before that make sure to exported the modified version of the catalogue (after answering question 2 of the proposed exercises. Be sure to practice how to plot data, as we will try some of the most common representations of the catalogues.